# Colab: оточення для 3D Model Soft

Цей ноутбук клонує проєкт, установлює його залежності та перевіряє NVIDIA GPU через PyTorch. Генерація Zero123++ буде окремим завданням.

1. Відкрийте ноутбук у Google Colab через **File → Upload notebook** або GitHub.
2. Виберіть **Runtime → Change runtime type → Python 3 → GPU** (наприклад, T4).
3. На свіжому runtime виконайте комірки зверху вниз (**Runtime → Run all**).
4. У конфігурації можна вибрати гілку, тег або SHA коміту. Для відтворюваного запуску збережіть надрукований SHA та використайте його як `REPO_REF`.

Потрібні інтернет, Git, Python ≥3.12 і встановлений у GPU runtime PyTorch із CUDA. Доступність GPU у Colab залежить від ресурсів облікового запису. Файли в `/content` тимчасові; збережіть потрібні результати перед завершенням сесії.

Джерела: [Colab FAQ](https://research.google.com/colaboratory/faq.html), [pip: запуск через Python](https://pip.pypa.io/en/stable/user_guide/#running-pip), [PyTorch: CUDA](https://docs.pytorch.org/docs/stable/notes/cuda.html).


In [ ]:
from pathlib import Path
import importlib
import importlib.metadata
import subprocess
import site
import sysconfig
import sys

REPO_URL = "https://github.com/Lenuar23/3d-model_generator.git"
REPO_REF = "main"  # Гілка, тег або SHA; для цієї задачі: chore/colab-setup після push.
REPO_DIR = Path("/content/3d-model_generator")

if sys.version_info < (3, 12):
    raise RuntimeError(
        "Проєкт потребує Python >=3.12. Виберіть сумісний Python runtime "
        "у Runtime → Change runtime type та запустіть ноутбук з початку."
    )
if not REPO_URL.strip() or not REPO_REF.strip() or REPO_REF.startswith("-"):
    raise ValueError("Укажіть непорожні REPO_URL і REPO_REF; ref не може починатися з '-'.")
print("Python:", sys.version.split()[0], "| kernel:", sys.executable)


## 1. Перевірка GPU

Використовуємо PyTorch із runtime без перевстановлення CUDA або драйверів. Успіх означає, що GPU доступний і виконав невелику операцію; ця перевірка не гарантує достатньої VRAM для майбутньої моделі.


In [ ]:
try:
    import torch
except ImportError as exc:
    raise RuntimeError(
        "PyTorch відсутній. Оберіть стандартний Colab GPU runtime "
        "через Runtime → Change runtime type і почніть зі свіжої сесії."
    ) from exc

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU недоступний. У Runtime → Change runtime type виберіть GPU, "
        "перепідключіть runtime та повторіть запуск. Якщо Colab не виділяє GPU, "
        "спробуйте пізніше; CPU/TPU не проходять цю перевірку."
    )

try:
    with torch.no_grad():
        probe = torch.tensor([1.0, 2.0, 3.0], device="cuda")
        result = (probe * probe).sum().item()
    torch.cuda.synchronize()
    if result != 14.0:
        raise RuntimeError(f"Неочікуваний результат CUDA: {result}")
except RuntimeError as exc:
    raise RuntimeError(
        "GPU знайдено, але CUDA-обчислення не вдалося. "
        "Перезапустіть свіжий GPU runtime та повторіть запуск."
    ) from exc

properties = torch.cuda.get_device_properties(0)
print("PyTorch:", torch.__version__, "| CUDA:", torch.version.cuda)
print("GPU:", properties.name, "| VRAM, GiB:", round(properties.total_memory / 1024**3, 2))
print("CUDA smoke check: OK, result =", result)
del probe


## 2. Клонування репозиторію

Повторний запуск використовує ту саму папку та вибирає актуальний commit заданого ref у detached HEAD. Якщо папка містить зміни або належить іншому репозиторію, комірка зупиниться. Збережіть зміни самостійно або задайте інший `REPO_DIR`; автоматичного видалення файлів немає.


In [ ]:
def git(*args, cwd=None):
    return subprocess.run(
        ["git", *map(str, args)], cwd=cwd, check=True,
        text=True, capture_output=True,
    ).stdout.strip()


def prepare_checkout(repo_url, repo_ref, repo_dir):
    repo_dir = Path(repo_dir).expanduser().resolve()
    if repo_dir.exists():
        if not (repo_dir / ".git").is_dir():
            raise RuntimeError(f"{repo_dir} вже існує і не є окремим Git checkout. Задайте іншу папку.")
        if git("remote", "get-url", "origin", cwd=repo_dir) != repo_url:
            raise RuntimeError("origin не збігається з REPO_URL. Задайте інший REPO_DIR.")
        if git("status", "--porcelain", "--untracked-files=all", cwd=repo_dir):
            raise RuntimeError("Checkout має локальні зміни. Збережіть їх або задайте інший REPO_DIR.")
    else:
        repo_dir.parent.mkdir(parents=True, exist_ok=True)
        git("clone", "--no-checkout", "--", repo_url, repo_dir)

    # FETCH_HEAD однозначно посилається на запитаний ref, без застарілої локальної гілки.
    git("fetch", "--depth", "1", "origin", repo_ref, cwd=repo_dir)
    commit = git("rev-parse", "--verify", "FETCH_HEAD^{commit}", cwd=repo_dir)
    git("checkout", "--detach", "--no-overwrite-ignore", commit, cwd=repo_dir)
    if not (repo_dir / "pyproject.toml").is_file():
        raise RuntimeError("У вибраному ref немає pyproject.toml. Перевірте REPO_REF.")
    print("Checkout:", repo_dir, "| commit:", commit)
    return repo_dir


REPO_DIR = prepare_checkout(REPO_URL, REPO_REF, REPO_DIR)


## 3. Залежності проєкту

Установлюємо пакет із вибраного checkout та залежності з його `pyproject.toml` через Python поточного kernel. Зараз це NumPy; залежності моделей додаватимуться в наступних завданнях. Цей запуск використовує обмеження версій із `pyproject.toml`, а не lock-файл.

Якщо NumPy вже був імпортований і його версія змінилася під час установки, комірка попросить **Runtime → Restart session**. Після перезапуску повторіть комірки з початку.


In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--editable", str(REPO_DIR)],
    check=True,
)
importlib.invalidate_caches()

loaded_numpy = sys.modules.get("numpy")
installed_numpy_version = importlib.metadata.version("numpy")
if loaded_numpy is not None and loaded_numpy.__version__ != installed_numpy_version:
    raise RuntimeError(
        "Версію NumPy оновлено після імпорту. Виконайте Runtime → Restart session "
        "і повторіть усі комірки з початку."
    )
# Обробляємо новий editable .pth у kernel без перезапуску.
site.addsitedir(sysconfig.get_paths()["purelib"])
print("Installed 3d-model-soft:", importlib.metadata.version("3d-model-soft"))
print("Installed NumPy:", installed_numpy_version)


## 4. Перевірка імпорту й залежностей

Імпортуємо встановлений пакет у поточному kernel і перевіряємо проєкцію початку координат. Через цифру на початку назви пакета використовуємо `importlib`. Фінальний успіх потребує проходження всіх попередніх комірок, зокрема CUDA-перевірки.


In [ ]:
import numpy as np

models = importlib.import_module("3d_model_soft.domain.models")
camera_math = importlib.import_module("3d_model_soft.services.camera_math")
package_path = Path(models.__file__).resolve()
if REPO_DIR / "src" not in package_path.parents:
    raise RuntimeError(
        f"Імпортовано інший checkout: {package_path}. "
        "Перезапустіть session і повторіть запуск з початку."
    )
camera = models.Camera(
    azimuth_deg=0, elevation_deg=0, distance=4, fov_deg=90,
    width=201, height=101,
)
pixels = camera_math.project(np.array([[0.0, 0.0, 0.0]]), camera)
np.testing.assert_allclose(pixels, [[100.0, 50.0]])
print("Project import / NumPy projection: OK")
print("Оточення готове. Збережіть SHA коміту та вивід перевірки GPU для звіту.")
